# 浩劫原生自对弈训练
在 Colab 选择 **L4 / A100 或其他支持原生 BF16 的 GPU**。T4、P100 不支持本训练链路。

依次安装、查看自动配置，再将启动单元的 `START_TRAINING` 改为 `True`。默认不启动训练。Python 只负责安装和控制，采集、推理、学习与保存全部运行在原生程序中。

源码与依赖放在 `/content`；运行目录和定时检查点放在 Google Drive。断开或回收运行时后，从最近的完整 `checkpoint.pt` 恢复到新目录。保存间隔默认 300 秒，也可手动保存或停止。

In [ ]:
from pathlib import Path
import json
import subprocess
from datetime import datetime, timezone

REPOSITORY = 'https://github.com/Loremmlel/haojie2.git'
REVISION = 'codex/bf16-elastic-training'  # 也可填已推送的提交号
SOURCE = Path('/content/haojie2')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], check=True)
subprocess.run(['bash', str(SOURCE / 'training/colab/setup.sh')], check=True)
NATIVE = ['bash', str(SOURCE / 'training/native/run.sh')]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/haojie-training')
RESUME = ''  # 恢复时填旧运行目录中的 checkpoint.pt 完整路径
OVERRIDES = []  # 例如 ['--concurrency', '16']；通常使用自动配置
RUN = ROOT / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
ARGS = ['--device', 'cuda', *OVERRIDES]
if RESUME:
    ARGS += ['--resume', RESUME]
configuration = json.loads(subprocess.check_output([*NATIVE, 'configure', *ARGS], text=True))
print(json.dumps(configuration, ensure_ascii=False, indent=2))
print('新运行目录:', RUN)

In [ ]:
START_TRAINING = False
if START_TRAINING:
    if 'process' in globals() and process.poll() is None:
        raise RuntimeError('已有训练进程运行，请先在控制单元停止。')
    if RUN.exists():
        raise RuntimeError('运行目录已存在，请重新执行配置单元生成新目录。')
    ROOT.mkdir(parents=True, exist_ok=True)
    LOG = RUN.with_suffix('.log')
    with LOG.open('x') as output:
        process = subprocess.Popen([*NATIVE, 'run', '--directory', str(RUN), *ARGS],
                                   stdout=output, stderr=subprocess.STDOUT)
    print('已启动，PID:', process.pid, '\n日志:', LOG)
else:
    print('尚未启动。确认配置后将 START_TRAINING 改为 True，再运行此单元。')

In [ ]:
ACTION = 'status'  # status 查看；save 保存；stop 保存并停止；resize 调整并发
CONCURRENCY = 8  # 仅在 ACTION == 'resize' 时使用
if ACTION not in {'status', 'save', 'stop', 'resize'}:
    raise ValueError('未知控制操作')
if not RUN.exists():
    print('尚未启动或正在初始化。')
elif ACTION == 'status':
    result = subprocess.run([*NATIVE, 'status', '--directory', str(RUN)], text=True, capture_output=True)
    print(json.dumps(json.loads(result.stdout), ensure_ascii=False, indent=2)
          if result.returncode == 0 else '正在初始化；如进程已退出，请查看日志。')
else:
    control = [*NATIVE, ACTION, '--directory', str(RUN)]
    if ACTION == 'resize':
        control += ['--concurrency', str(CONCURRENCY)]
    subprocess.run(control, check=True)
    print('请求已送达；save 完成以 saved.json 更新时间为准。')
    if ACTION == 'stop' and 'process' in globals():
        print('退出码:', process.wait())
        print('恢复检查点:', RUN / 'checkpoint.pt')
if 'process' in globals() and process.poll() is not None:
    print('进程已退出，退出码:', process.returncode)
if RUN.with_suffix('.log').exists():
    with RUN.with_suffix('.log').open('rb') as log:
        log.seek(max(0, log.seek(0, 2) - 4000))
        print(log.read().decode('utf-8', errors='replace'))